# Q15: Cross-Encoders for Re-ranking
**Topic:** LLM | **Difficulty:** Medium | **Time:** 15 min
**Sheet:** GrindLLM50

---

## Question
Why are cross-encoders expensive? How do you prepare a re-ranker for production?

## Detailed Answer

### Why Cross-Encoders Are Expensive
A cross-encoder takes BOTH query and document as input:
```
[CLS] query_tokens [SEP] document_tokens [SEP] → Relevance Score
```

- Must process query + document **together** through the full Transformer
- For N documents: N separate forward passes (no caching)
- Compare: Bi-encoder encodes query once, documents once → dot product similarity

| Approach | Encode | Compare 1000 docs | Quality |
|----------|--------|-------------------|--------|
| Bi-encoder | 1 + 1000 passes | 1000 dot products | Good |
| Cross-encoder | 1000 joint passes | N/A (built-in) | Best |

### Why Better Quality?
Cross-encoders enable **cross-attention** between query and document tokens.
- Can capture fine-grained interactions
- Bidirectional context-aware matching
- Bi-encoders compress everything to fixed-size vectors → information loss

### Production Re-ranking Pipeline
```
Query → Bi-encoder retrieval (fast, top-100) → Cross-encoder re-ranking (slow, top-10) → User
```

### Making It Production-Ready
1. **Two-stage pipeline**: Bi-encoder filters thousands → cross-encoder scores top-K (K=20-100)
2. **Distillation**: Train smaller cross-encoder from large one
3. **Quantization**: INT8 the cross-encoder
4. **Batching**: Process all query-doc pairs in a batch for GPU efficiency
5. **Caching**: Cache popular query-document scores
6. **ONNX export**: Faster inference than PyTorch
7. **Async processing**: Score documents asynchronously, return as ready

### Training a Re-ranker
- **Data**: (query, positive_doc, negative_doc) triplets
- **Loss**: Binary cross-entropy or pairwise ranking loss
- **Model**: Fine-tuned BERT/DeBERTa (cross-encoder style)
- **Hard negatives**: Use bi-encoder to find hard negatives (close but wrong)

## Key Takeaways
- Cross-encoders attend jointly to query + document → **better quality, much slower**
- Production pattern: **bi-encoder (fast recall) → cross-encoder (precise re-ranking)**
- Limit cross-encoder to **top 20-100 results** for latency control
- **Distillation + quantization + ONNX** for production speed